# MergeKit-Paper-Repro: One-Click Azure ML Reproduction

Reproduces the flagship case study from the original MergeKit paper
(Goddard et al. 2024, [arXiv 2403.13257](https://arxiv.org/abs/2403.13257),
Table 1): merging **Llama-2-7B-Chat** + **Meditron-7B** (a medical-domain
fine-tune) via four methods — **LERP, SLERP, TIES, DARE-TIES** — and
evaluating all of it plus both source checkpoints on six benchmarks (USMLE,
MedMCQA, PubMedQA, ARC-Challenge, HellaSwag, MMLU), directly comparable to
the paper's own published numbers.

**This notebook runs the split pipeline by default** — 4 separate merge
jobs, then 6 separate eval jobs, all on T4 (`gpu-cluster-merge`), never
combined into one job. That's not a stylistic choice: the combined
one-job-does-everything design reliably exhausts local disk (all 4 merged
7B outputs resident at once, ~52GB, right as evaluation starts loading a
5th model on top) and has failed this way on a genuinely fresh environment.
Splitting removes the failure mode structurally instead of working around
it after the fact. See the Appendix for the full bug history that led here.

Once you provide an HF token, running the whole reproduction is one
function call: `run_full_reproduction(HF_TOKEN)`. It submits all 4 merges
in parallel, waits, registers each output as a data asset, submits all 6
evals in parallel, waits, then downloads and compares every result against
the paper automatically.

Repo: `alan-turing-institute/model-merging`, folder
`merge-job-mergekit-paper-repro/` (this notebook lives there too — the
job `.yml` files referenced below are sibling files, no `cd` needed).


## Prerequisites

1. **Azure CLI with the `ml` extension**, logged into an account with a role
   on the `TIRE-1` resource group / `TIRE-2` workspace.
2. **A Hugging Face access token** (Settings → Access Tokens → New token,
   **Read** scope is enough).
3. **Access to the gated `epfl-llm/meditron-7b` repo** — visit its model
   page on huggingface.co while logged into the account that owns your
   token, and accept its terms. Without this, every job touching
   Meditron-7B fails with a 401, regardless of how valid the token
   otherwise is.


In [ ]:
import os

# Jupyter kernels don't inherit shell customizations from ~/.zshrc/~/.bash_profile,
# so `!az`/subprocess calls to `az` can fail with "command not found" even if
# it works fine in a terminal. az is installed in a venv here rather than on
# the global PATH -- add it once, for this kernel session.
AZ_VENV_BIN = "/Users/mpietrzyk/azure-cli-venv/bin"
if AZ_VENV_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = AZ_VENV_BIN + os.pathsep + os.environ["PATH"]

!az version

## Repo structure

| Category | Files | What it does |
|---|---|---|
| **Merge-only** (default, used by this notebook) | `job-merge-single-{lerp,slerp,ties,dare-ties}.yml` + `merge_single.sh` | Merges exactly one method, writes to a `uri_folder` output, no eval |
| **Eval-only** (default, used by this notebook) | `job-eval-single-{lerp,slerp,ties,dare-ties,llama2,meditron}.yml` + `eval_single.sh` | Evaluates exactly one target — mounts a merged-model data asset read-only, or pulls a base checkpoint from HF Hub |
| **Combined** (legacy, not used by default) | `job.yml` + `run_all.sh`, `job-dare-ties-fixed.yml` | Merges all 4 methods and evaluates all 6 targets in one job. Kept for reference; this is the design that exhausts disk. |
| **Maintenance** | `job-patch-lerp-config.yml` + `patch_config.sh` | Downloads a merge, patches a config bug, re-uploads as a new data asset version |

No job in the default pipeline ever does more than one merge or one eval,
and no job ever holds more than one 7B model's worth of local disk.


In [ ]:
import subprocess

def check_meditron_access(token):
    """Returns True if `token` genuinely has access to the gated
    epfl-llm/meditron-7b repo -- checks the real file-resolve endpoint.
    /api/models/{repo} returns 200 for anyone, authenticated or not, and
    does NOT actually validate gated access -- don't use it for this check."""
    result = subprocess.run(
        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
         "-H", f"Authorization: Bearer {token}",
         "https://huggingface.co/epfl-llm/meditron-7b/resolve/main/config.json"],
        capture_output=True, text=True,
    )
    status = result.stdout.strip()
    print(f"HTTP status: {status}")
    return status == "200"

# Set your real token below (not a placeholder like <your-token> -- angle
# brackets are shell redirection characters and will crash job submission
# if left in literally; a plain-text mistake is much safer to catch).
HF_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

assert HF_TOKEN != "REPLACE_WITH_YOUR_TOKEN", "Set HF_TOKEN to your real token first."
assert check_meditron_access(HF_TOKEN), "Token does not have access to epfl-llm/meditron-7b -- check huggingface.co/settings/gated-repos"
print("Token OK, has Meditron-7B access.")

## Orchestration helpers

Everything below runs `az` via `subprocess`, in your own terminal session
under your own credentials — nothing here is executed by an AI assistant on
your behalf, it's plain Python you're about to run yourself.


In [ ]:
import subprocess
import json
import time

RG = "TIRE-1"
WS = "TIRE-2"

def _az(args):
    """Run an az command, return parsed JSON if possible else raw stdout text."""
    cmd = ["az"] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"az command failed: {' '.join(cmd)}\n{result.stderr}")
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return result.stdout.strip()

def submit_job(yaml_file, set_args=None):
    """Submit a job from a yaml file, optionally with --set key=value pairs
    (e.g. {"inputs.hf_token": token}). Returns the job name."""
    args = ["ml", "job", "create", "-f", yaml_file,
            "--resource-group", RG, "--workspace-name", WS]
    for k, v in (set_args or {}).items():
        args += ["--set", f"{k}={v}"]
    args += ["--query", "name", "-o", "tsv"]
    name = _az(args)
    print(f"Submitted {yaml_file} -> {name}")
    return name

def job_status(name):
    return _az(["ml", "job", "show", "--name", name,
                "--resource-group", RG, "--workspace-name", WS,
                "--query", "status", "-o", "tsv"])

def wait_for_jobs(names_by_label, poll_seconds=30, timeout_seconds=10800):
    """Poll a dict of {label: job_name} until every job reaches a terminal
    state. Returns {label: final_status}."""
    remaining = dict(names_by_label)
    final = {}
    start = time.time()
    while remaining:
        for label, name in list(remaining.items()):
            status = job_status(name)
            if status in ("Completed", "Failed", "Canceled"):
                print(f"  {label} ({name}): {status}")
                final[label] = status
                del remaining[label]
        if remaining:
            if time.time() - start > timeout_seconds:
                raise TimeoutError(f"Timed out waiting for: {list(remaining.keys())}")
            time.sleep(poll_seconds)
    return final

def register_data_asset(name, job_name, output_name):
    """Register a completed job's output as a new version of a named data asset."""
    path = f"azureml://datastores/workspaceblobstore/paths/azureml/{job_name}/{output_name}"
    result = _az(["ml", "data", "create", "--name", name, "--type", "uri_folder",
                  "--path", path, "--resource-group", RG, "--workspace-name", WS,
                  "-o", "json"])
    version = result["version"]
    print(f"Registered {name} v{version}")
    return version

def download_results(job_name, local_dir):
    subprocess.run(
        ["az", "ml", "job", "download", "--name", job_name,
         "--resource-group", RG, "--workspace-name", WS,
         "--download-path", local_dir, "--output-name", "results"],
        capture_output=True, text=True,
    )

def wait_for_job(name, poll_seconds=30, timeout_seconds=10800):
    """Convenience wrapper around wait_for_jobs for a single job."""
    return wait_for_jobs({"_single": name}, poll_seconds=poll_seconds, timeout_seconds=timeout_seconds)["_single"]

In [ ]:
MERGE_JOBS = {
    "lerp": "job-merge-single-lerp.yml",
    "slerp": "job-merge-single-slerp.yml",
    "ties": "job-merge-single-ties.yml",
    "dare-ties": "job-merge-single-dare-ties.yml",
}

EVAL_JOBS = {
    "lerp": "job-eval-single-lerp.yml",
    "slerp": "job-eval-single-slerp.yml",
    "ties": "job-eval-single-ties.yml",
    "dare-ties": "job-eval-single-dare-ties.yml",
    "llama2-base": "job-eval-single-llama2.yml",
    "meditron-base": "job-eval-single-meditron.yml",
}

# Only these eval jobs declare an hf_token input at all -- passing --set for
# one that doesn\'t declare it is a hard error, not a harmless no-op.
EVAL_JOBS_NEEDING_TOKEN = {"meditron-base"}

PAPER_MODEL_KEY = {
    "lerp": "merged-lerp", "slerp": "merged-slerp", "ties": "merged-ties",
    "dare-ties": "merged-dare-ties", "llama2-base": "llama2-7b-chat",
    "meditron-base": "meditron-7b",
}

## Results analysis helper

In [ ]:
import glob

PAPER = {
    "llama2-7b-chat": {"medqa_4options": 35.90, "medmcqa": 35.45, "pubmedqa": 73.40, "arc_challenge": 44.20, "hellaswag": 55.40, "mmlu": 46.37},
    "meditron-7b":     {"medqa_4options": 38.40, "medmcqa": 24.07, "pubmedqa": 71.40, "arc_challenge": 40.20, "hellaswag": 54.50, "mmlu": 33.06},
    "merged-lerp":      {"medqa_4options": 39.10, "medmcqa": 36.65, "pubmedqa": 75.60, "arc_challenge": 46.76, "hellaswag": 58.66, "mmlu": 48.44},
    "merged-slerp":     {"medqa_4options": 39.20, "medmcqa": 36.91, "pubmedqa": 75.60, "arc_challenge": 46.84, "hellaswag": 58.67, "mmlu": 47.97},
    "merged-ties":      {"medqa_4options": 38.73, "medmcqa": 32.27, "pubmedqa": 75.60, "arc_challenge": 45.05, "hellaswag": 58.23, "mmlu": 45.03},
    "merged-dare-ties": {"medqa_4options": 36.37, "medmcqa": 27.56, "pubmedqa": 72.20, "arc_challenge": 42.92, "hellaswag": 54.79, "mmlu": 41.17},
}
METRIC = {"medqa_4options": "acc", "medmcqa": "acc", "pubmedqa": "acc",
          "arc_challenge": "acc_norm", "hellaswag": "acc_norm", "mmlu": "acc"}

def load_and_compare(results_dir, model_key):
    """Load a downloaded job's results_*.json and compare against PAPER[model_key].
    Returns mean |diff| excluding hellaswag (a known model-independent offset)."""
    files = glob.glob(f"{results_dir}/**/results_*.json", recursive=True)
    if not files:
        raise FileNotFoundError(f"No results_*.json under {results_dir}")
    data = json.load(open(sorted(files)[-1]))
    paper = PAPER[model_key]
    diffs = []
    header_task, header_ours, header_paper, header_diff = "Task", "Ours", "Paper", "Diff"
    print(f"{header_task:15s} {header_ours:>8s} {header_paper:>8s} {header_diff:>8s}")
    for task, paper_val in paper.items():
        r = data["results"].get(task, {})
        m = METRIC[task]
        val = r.get(f"{m},none", r.get(m))
        if val is None:
            print(f"{task:15s} {'MISSING':>8s}")
            continue
        val *= 100
        diff = val - paper_val
        if task != "hellaswag":
            diffs.append(abs(diff))
        print(f"{task:15s} {val:8.2f} {paper_val:8.2f} {diff:+8.2f}")
    mean_diff = sum(diffs) / len(diffs)
    print(f"mean |diff| (excl hellaswag): {mean_diff:.2f}")
    return mean_diff

## LERP config-patch helper

LERP's merge has a known, deterministic bug (see the gotchas below): Llama-2-7B-Chat (vocab_size=32000) and Meditron-7B (vocab_size=32017) merge to a `config.json` that can declare the *wrong* vocab_size. It has now reproduced across two independent fresh merges, so `run_eval_only` (and therefore `run_full_reproduction`, which delegates to it) always patches the current LERP merge before evaluating it -- patching an already-correct `config.json` is a no-op, so this is safe to run unconditionally every time, not just when the bug is suspected.

`patch_lerp_and_reeval` below is the standalone version, for when LERP's eval already ran against an unpatched merge and came back near-random -- no need to re-run the other 5 targets or re-merge anything.

In [ ]:
def patch_lerp_config():
    """Patch the current mergekit-repro-lerp asset's config.json
    vocab_size bug and register the fix as a new asset version.
    LERP-specific -- SLERP/TIES/DARE-TIES have not shown this bug
    across two independent fresh merges. Safe to call even if the
    current version is already patched (patching 32000 -> 32000 is a
    no-op)."""
    print("=== Patching LERP config.json vocab_size ===")
    patch_job = submit_job("job-patch-lerp-config.yml")
    status = wait_for_job(patch_job)
    if status != "Completed":
        raise RuntimeError(f"LERP config patch job ended with status {status}")
    version = register_data_asset("mergekit-repro-lerp", patch_job, "patched_model")
    print(f"mergekit-repro-lerp patched -> v{version}")
    return version


def patch_lerp_and_reeval(hf_token=None):
    """Patch the current LERP merge's config bug and re-run just its
    eval. hf_token is accepted for signature consistency with the
    other run_* functions but currently unused -- neither the patch
    job nor the LERP eval job touches HF Hub (LERP evaluates a
    locally-mounted merge, not a base checkpoint)."""
    patch_lerp_config()

    print("\n=== Re-submitting LERP eval against the patched version ===")
    eval_name = submit_job(EVAL_JOBS["lerp"])
    status = wait_for_job(eval_name)
    if status != "Completed":
        print(f"WARNING: LERP eval ended with status {status}")
        return eval_name, None

    local_dir = "results/lerp"
    download_results(eval_name, local_dir)
    print("\n--- lerp (patched) ---")
    diff = load_and_compare(local_dir, PAPER_MODEL_KEY["lerp"])
    return eval_name, diff

## One-click reproduction

Submits all 4 merges in parallel (T4), waits, registers each output as a
data asset, submits all 6 evals in parallel (T4), waits, then downloads and
compares every result against the paper.


In [ ]:
def run_eval_only(hf_token):
    """Submit and wait for all 6 eval jobs against whatever data assets are
    currently registered. Use this directly (skipping run_full_reproduction)
    when the merges already succeeded and were registered in an earlier
    call/session, and only the eval step needs re-running -- e.g. after a
    partial failure like a CUDA OOM on the eval jobs specifically."""
    patch_lerp_config()  # idempotent -- safe even if already patched

    print("=== Submitting eval jobs (6x T4, parallel) ===")
    eval_names = {}
    for target, f in EVAL_JOBS.items():
        set_args = {"inputs.hf_token": hf_token} if target in EVAL_JOBS_NEEDING_TOKEN else None
        eval_names[target] = submit_job(f, set_args)

    print("\n=== Waiting for evals ===")
    eval_status = wait_for_jobs(eval_names)

    print("\n=== Downloading and comparing results ===")
    results = {}
    for target, name in eval_names.items():
        if eval_status[target] != "Completed":
            print(f"\n--- {target}: skipped (status={eval_status[target]}) ---")
            continue
        local_dir = f"results/{target}"
        download_results(name, local_dir)
        print(f"\n--- {target} ---")
        try:
            results[target] = load_and_compare(local_dir, PAPER_MODEL_KEY[target])
        except Exception as e:
            print(f"Could not parse results for {target}: {e}")

    print("\n=== Summary: mean |diff| vs paper ===")
    for target, diff in results.items():
        print(f"{target:15s} {diff:.2f}")

    return eval_names, results


def run_full_reproduction(hf_token):
    print("=== Step 1/3: submitting merge jobs (4x T4, parallel) ===")
    merge_names = {m: submit_job(f, {"inputs.hf_token": hf_token}) for m, f in MERGE_JOBS.items()}

    print("\n=== Step 2/3: waiting for merges ===")
    merge_status = wait_for_jobs(merge_names)
    failed = [m for m, s in merge_status.items() if s != "Completed"]
    if failed:
        raise RuntimeError(f"Merge job(s) did not complete: {failed}")

    print("\n=== registering merged models as data assets ===")
    for method, name in merge_names.items():
        register_data_asset(f"mergekit-repro-{method}", name, "merged_model")

    print("\n=== Step 3/3: running eval ===")
    eval_names, results = run_eval_only(hf_token)

    return merge_names, eval_names, results

# One-click full run:
# merge_jobs, eval_jobs, results = run_full_reproduction(HF_TOKEN)
#
# Or, if merges are already registered from an earlier run and only eval
# needs (re-)running:
# eval_jobs, results = run_eval_only(HF_TOKEN)
print("Run one of the commented lines above once HF_TOKEN is set and verified.")

## Known gotchas baked into this design

These aren't hypothetical — each one actually happened across two
independent runs of this reproduction, and each shaped a specific design
choice above.

**`${{inputs.X}}` doesn't reliably resolve inside `environment_variables:`.**
An earlier version of these jobs passed the HF token via
`environment_variables: {HF_TOKEN: ${{inputs.hf_token}}}`. Six consecutive
submissions failed with an identical 401 on `epfl-llm/meditron-7b`, even
with a confirmed-valid token each time — the container's `HF_TOKEN` was
literally the unresolved string `${{inputs.hf_token}}`. Fix: pass it as a
positional argument in `command:` instead (the context already proven to
work for every `uri_folder` input in this repo), and `export HF_TOKEN="$N"`
inside the script.

**The wrong HF endpoint silently validates nothing.**
`https://huggingface.co/api/models/{repo}` returns `200` for anyone,
authenticated or not — it's public metadata and never checks gated access.
A literal, unreplaced placeholder token passed this check and then failed
downstream. `check_meditron_access()` above uses the real endpoint
(`/resolve/main/config.json`) instead.

**Combined merge+eval exhausts disk.**
Keeping all 4 merged 7B outputs resident locally (~52GB) before evaluation
starts loading a 5th model filled a 63GB container disk to 86% right as the
first eval began. This is why merging and evaluating are separate jobs by
default here, each reading/writing at most one model's worth of local data.

**The LERP vocab-mismatch bug reappears on every fresh merge.**
Llama-2-7B-Chat (vocab_size=32000) and Meditron-7B (vocab_size=32017) merge
to a `config.json` that can carry the *wrong* vocab_size. The tempting
`ignore_mismatched_sizes=True` workaround is lossy — it discards and
randomly reinitializes the *entire* mismatched tensor, not just the ~17
actually-wrong rows, producing near-random scores (MMLU dropped to 22.99%
in one run). The real fix (`job-patch-lerp-config.yml`) patches `vocab_size` directly and re-registers a corrected data asset version. This reproduced a second time on an independent fresh merge, so `run_eval_only`/`run_full_reproduction` now call `patch_lerp_config()` unconditionally before ever evaluating LERP, rather than only after noticing bad results.

**`rw_mount` is not a valid mode for job *inputs*** — only `download`,
`ro_mount`, `direct` are; `rw_mount` is output-only. The patch job above
has to download-then-copy-to-a-writable-output rather than mounting the
existing asset read-write in place.

**Angle-bracket placeholders can crash bash outright.**
`<your-token>` contains shell redirection characters. If left unsubstituted
in a real submission, bash tries to parse `<`/`>` as I/O redirects and
crashes with a syntax error before the job even starts. `HF_TOKEN` above
uses `REPLACE_WITH_YOUR_TOKEN` specifically to avoid this trap, and the
`assert` catches an unreplaced value before submission.


## Results from a completed reproduction (real data)

A full split-job run against all 6 targets, compared to a prior independent
run of the same experiment three weeks earlier:

| Target | This run | Prior run | Delta |
|---|---|---|---|
| llama2-base | 1.22 | 1.23 | +0.01 (rounding only) |
| meditron-base | 3.36 | 3.37 | +0.01 (rounding only) |
| LERP | 1.34 | 1.34 | +0.00 |
| SLERP | 1.39 | 1.39 | +0.00 |
| TIES | 1.35 | 1.35 | +0.00 |
| DARE-TIES | 1.99 | 2.88 | −0.89 |

Five of six targets are essentially perfectly reproducible across two
independent runs — expected, since none of them involve any randomness in
either the merge or the loglikelihood-based eval. DARE-TIES is the one
genuine source of run-to-run variance, fully explained rather than
mysterious: MergeKit's DARE dropout mask (`torch.bernoulli`) is unseeded by
default, so every DARE-TIES run is one noisy draw, not a fixed number.

Per-benchmark breakdown (raw scores, this run):

| Benchmark | llama2-base | meditron-base | LERP | SLERP | TIES | DARE-TIES |
|---|---|---|---|---|---|---|
| USMLE (medqa) | 39.28 | 27.26 | 41.71 | 41.79 | 35.35 | 33.70 |
| MedMCQA | 37.41 | 26.63 | 40.57 | 40.57 | 34.50 | 29.57 |
| PubMedQA | 73.40 | 70.60 | 75.60 | 75.60 | 76.00 | 73.40 |
| ARC-Challenge | 43.52 | 41.89 | 46.67 | 46.50 | 45.56 | 44.20 |
| HellaSwag | 75.98 | 72.93 | 76.81 | 76.90 | 75.62 | 73.23 |
| MMLU | 46.47 | 32.43 | 48.37 | 48.33 | 45.26 | 38.36 |

Every single target lands +17 to +21 points above the paper on HellaSwag,
including both unmerged source checkpoints — a uniform, model-independent
offset (most likely a few-shot count difference) rather than anything
about the merges. Meditron-base's `medqa` result (27.26 vs. paper's 38.40,
a genuine −11.14 outlier) stands alone as the only cell in this whole table
beyond roughly ±4 points.


## Appendix — full bug log for this reproduction

In the order they were hit, across the original run and the independent
re-run that led to this notebook's design:

1. **`huggingface_hub>=1.16`** tightened its `hf://` URI parser, breaking
   `datasets`' loader for bare-slug canonical datasets (`medmcqa`) —
   fixed by pinning `huggingface_hub<1.16` in the Dockerfile.
2. **`datasets>=4.0`** removed script-based dataset loaders entirely,
   breaking `pubmedqa` (`bigbio/pubmed_qa` ships a builder script) —
   fixed by pinning `datasets<4.0.0`.
3. **`pubmedqa` still needs `trust_remote_code`** even with the pin above
   — `lm_eval`'s own model-arg doesn't propagate to dataset loading (a
   known open upstream issue). Fixed via `HF_DATASETS_TRUST_REMOTE_CODE=1`.
   Re-introduced once (forgot to carry it into the first split-job
   scripts) and hit all 6 targets again before being fixed a second time.
4. **Vocab-size mismatch** (Llama-2-7B-Chat=32000 vs. Meditron-7B=32017)
   breaks LERP's `config.json` metadata specifically, on every fresh merge
   — see the gotcha above.
5. **`dare_ties`'s default `rescale=True`** can overflow `float16` to NaN
   during merge for vocab-mismatched tensors specifically — fixed via
   `dtype: bfloat16` in the merge config.
6. **`${{inputs.X}}` templating doesn't reliably resolve inside
   `environment_variables:`** — see the gotcha above.
7. **The wrong HF endpoint for verifying gated access** — see the gotcha
   above.
8. **Combined merge+eval disk exhaustion** — see the gotcha above; this is
   why the pipeline is split by default now.
9. **`rw_mount` is not a valid mode for job *inputs*** — see the gotcha
   above.
10. **Angle-bracket placeholders crashing bash** — see the gotcha above.
